In [3]:
import sys
if '/disks/cosmodm/vdvuurst' not in sys.path:
    sys.path.append('/disks/cosmodm/vdvuurst')

import numpy as np
import json
from functional_forms import *
import os
from itertools import *
from inspect import signature


In [4]:
all_funcs = [constant_func, linear_func, parabola_func, exponential_func, exponential_squared_func, power_law_func, power_linear_func, inverse_func, poly_4_func]
func_names = [f.__name__.split('_func')[0] for f in all_funcs]

# get_func_names = lambda flist: [f.__name__ for f in flist]
get_func_names = lambda flist: [f.__name__.split('_func')[0] for f in flist]

In [36]:
lambda_r_funcs = [exponential_func,inverse_func]
sigma_1_r_funcs = [poly_3_func, poly_4_func]
sigma_2_r_funcs = [linear_func, parabola_func]

m_funcs = [linear_func, poly_3_func, exponential_func]

no_params = lambda f: len(signature(f).parameters) - 1
no_params(exponential_func)

def flatten(xss):
    return [x for xs in xss for x in xs]

def create_function_combinations(funclist): 
    func_combis = [list(product([rfunc], list(combinations_with_replacement(m_funcs, no_params(rfunc))))) for rfunc in funclist]
    return flatten(func_combis)

lambda_funcs = create_function_combinations(lambda_r_funcs)
sigma_1_funcs = create_function_combinations(sigma_1_r_funcs)
sigma_2_funcs = create_function_combinations(sigma_2_r_funcs)

all_combis = list(product(lambda_funcs, sigma_1_funcs, sigma_2_funcs))
print(f'There are {len(all_combis)} function combinations')


There are 9216 function combinations


In [35]:
combi = all_combis[3598]

def no_params_in_combi(combi):
    n = 0
    m = 0
    for gauss_param in combi:
        r_form, m_forms = gauss_param
        for f in m_forms:
            print(f)
            n += len(signature(f).parameters) -1
        print(r_form, n)
        print()
        m += n
        n = 0
    return m

no_params_in_combi(combi)

<function poly_3_func at 0x7f254a8862a0>
<function poly_3_func at 0x7f254a8862a0>
<function exponential_func at 0x7f254a885f80>
<function exponential_func at 0x7f254a885f80> 11

<function linear_func at 0x7f254a885e40>
<function linear_func at 0x7f254a885e40>
<function exponential_func at 0x7f254a885f80>
<function exponential_func at 0x7f254a885f80>
<function exponential_func at 0x7f254a885f80>
<function poly_4_func at 0x7f254a886340> 13

<function poly_3_func at 0x7f254a8862a0>
<function poly_3_func at 0x7f254a8862a0>
<function exponential_func at 0x7f254a885f80>
<function parabola_func at 0x7f254a885ee0> 11



35

In [ ]:
with open('/disks/cosmodm/vdvuurst/data/initial_params_per_massbin.json', 'r') as file:
    initial_params_full = json.load(file)

mass_edges = [12 + 0.5*i for i in range(8)]
massbins = [f'M_{mass_edges[m]}-{mass_edges[m+1]}' for m in range(len(mass_edges)-1)]

# for key in massbins:
#     if key in ['M_12.0-12.5','M_12.5-13.0','M_15.0-15.5']: # these are all 0 since Sowmya didn't do them before
#         mkey = 'M_13.5-14.0' # so we set them to this massbin's values instead
#     else:
#         mkey = key
#     initial_params_full[key] = initial_params_full[mkey]


n = 0
average_values = np.zeros(9, dtype=np.float64)
for key, values in initial_params_full.items():
    print(key,values)
    if key in ['M_12.0-12.5','M_12.5-13.0','M_15.0-15.5']: # these are all 0 since Sowmya didn't do them before
        continue
    
    average_values += np.array(list(values.values()))
    n += 1

average_values /= n

init_param_dict_avg = {k:float(avg) for k,avg in zip(values.keys(),average_values)}

with open('/disks/cosmodm/vdvuurst/data/initial_params.json', 'w') as file:
    json.dump(init_param_dict_avg, file, indent = 1)

# initial params based on sowmya's results and for the new function based on those
# need only be in the ballpark
power_linear_init = np.array([init_param_dict_avg[x] for x in ['p', 'n', 'q', 'b']])
linear_init = np.array([init_param_dict_avg[x] for x in ['m', 'c']])
exp_init = np.array([init_param_dict_avg[x] for x in ['A', 'B', 'C']])
exp_sq_init = exp_init.copy()
power_law_init = power_linear_init.copy()[:-1]
const_init = 400.
parabola_init = np.hstack(([1], linear_init))
#the three below empircally deduced to work decently
inverse_init = np.array([1., 0.2]) 
poly_3_init = np.array([10, -50, 100, 50])
poly_4_init = np.array([10, -50, 100, 0.5, 10])

M_12.0-12.5 {'p': 0.0, 'n': 0.0, 'q': 0.0, 'b': 0.0, 'm': 0.0, 'c': 0.0, 'A': 0.0, 'B': 0.0, 'C': 0.0}
M_12.5-13.0 {'p': 0.0, 'n': 0.0, 'q': 0.0, 'b': 0.0, 'm': 0.0, 'c': 0.0, 'A': 0.0, 'B': 0.0, 'C': 0.0}
M_13.0-13.5 {'p': -227.7, 'n': 0.22, 'q': 37.6, 'b': 396.4, 'm': 7.8, 'c': 70.9, 'A': 0.69, 'B': 20, 'C': -0.005}
M_13.5-14.0 {'p': 511.3, 'n': 1.15, 'q': -653.4, 'b': 437, 'm': -9.42, 'c': 104.7, 'A': 0.22, 'B': 3.03, 'C': -0.009}
M_14.0-14.5 {'p': -505.3, 'n': 0.19, 'q': 40, 'b': 921.9, 'm': -4.6, 'c': 125.7, 'A': 0.1, 'B': 1, 'C': -0.015}
M_14.5-15.0 {'p': -554.6, 'n': 0.7, 'q': 262.4, 'b': 1007.8, 'm': -7.9, 'c': 140, 'A': 0.046, 'B': 0.45, 'C': -0.005}
M_15.0-15.5 {'p': 0.0, 'n': 0.0, 'q': 0.0, 'b': 0.0, 'm': 0.0, 'c': 0.0, 'A': 0.0, 'B': 0.0, 'C': 0.0}


array([  1.   ,  -3.53 , 110.325])